# MDIT RLBench Lab

faithful `MDIT` 单任务 `unplug_charger` 实验控制台。

这个 notebook 只负责拼接和调用 `.py` 入口，不直接承载模型实现。

In [ ]:
from pathlib import Path
import json
import os

PROJECT_ROOT = Path('/home/gjw/MyProjects/autodl_unplug_charger_transformer_fm')
PYTHON = '/home/gjw/miniconda3/envs/mdit_env/bin/python'
CONFIG = PROJECT_ROOT / 'configs' / 'mdit' / 'faithful_baseline.json'
RUN_DIR = PROJECT_ROOT / 'ckpt' / '<mdit_run_name>'
CKPT_PATH = RUN_DIR / 'best_success.pt'

print('project_root =', PROJECT_ROOT)
print('config =', CONFIG)

In [ ]:
train_cmd = f"""
{PYTHON} {PROJECT_ROOT / 'scripts' / 'run_mdit_autoresearch_trial.py'} \
  --phase train-only \
  --config {CONFIG} \
  --stage-epochs 100 \
  --checkpoint-every 100 \
  --experiment-name mdit_faithful_baseline_100
""".strip()

print(train_cmd)

In [ ]:
audit_cmd = f"""
{PYTHON} {PROJECT_ROOT / 'scripts' / 'run_mdit_autoresearch_trial.py'} \
  --phase audit-only \
  --run-dir {RUN_DIR} \
  --eval-episodes 20 \
  --audit-timeout-sec 10800 \
  --headless
""".strip()

eval_cmd = f"""
{PYTHON} {PROJECT_ROOT / 'scripts' / 'eval_mdit_checkpoint.py'} \
  --ckpt-path {CKPT_PATH} \
  --episodes 100 \
  --max-steps 200 \
  --headless
""".strip()

video_cmd = f"""
{PYTHON} {PROJECT_ROOT / 'scripts' / 'record_mdit_rollout_videos.py'} \
  --ckpt-path {CKPT_PATH} \
  --episodes 1 \
  --camera front \
  --output-dir {PROJECT_ROOT / 'ckpt' / 'videos' / 'mdit_demo'} \
  --no-headless
""".strip()

print('audit command:\n', audit_cmd)
print('\ncheckpoint eval command:\n', eval_cmd)
print('\nvideo command:\n', video_cmd)

In [ ]:
summary_path = RUN_DIR / 'summary.json'
audit_path = RUN_DIR / 'audit_report.json'

for path in [summary_path, audit_path]:
    if path.exists():
        print(f'===== {path.name} =====')
        print(json.dumps(json.loads(path.read_text()), indent=2, ensure_ascii=False)[:4000])
    else:
        print(f'missing: {path}')